# Notebook 4: Shallow NN with optimizer comparison (M4b — non-convex)

Hand-rolled 1-hidden-layer MLP on the heart-disease data, trained with the same seven optimizers from notebook 03. The hidden ReLU layer makes the loss surface non-convex, so optimizers should now genuinely separate (not just race to the same minimum).

Architecture: 17 inputs → 64 ReLU → 1 sigmoid (logistic output).
Loss: binary cross-entropy.

Same data, same train/val split, same lr-sweep protocol as notebook 03.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

SPLITS_DIR = '/content/drive/MyDrive/ECE567_Final/splits'
SEED = 2540
rng_init = np.random.default_rng(SEED)

data = np.load(os.path.join(SPLITS_DIR, 'split_preproc.npz'), allow_pickle=True)
X_tr, X_val = data['X_tr'], data['X_val']
y_tr, y_val = data['y_tr'].astype(np.float64), data['y_val'].astype(np.float64)

with open(os.path.join(SPLITS_DIR, 'results_baselines.json')) as fh:
    LBFGS_AUC = json.load(fh)['sklearn_lbfgs']['val_auc']

print('train:', X_tr.shape, 'val:', X_val.shape)
print(f'L-BFGS (linear) reference val AUC = {LBFGS_AUC:.5f}')

### MLP forward / backward (manual)

Let $H$ = hidden width, $D$ = input dim. With He initialization $W_1 \sim \mathcal{N}(0, 2/D)$.

Forward:
$$
h = \mathrm{ReLU}(X W_1 + b_1), \qquad z = h W_2 + b_2, \qquad p = \sigma(z).
$$

Backprop (per minibatch, $N$ rows):
$$
\delta_z = (p - y)/N, \quad \nabla_{W_2} = h^\top \delta_z, \quad \nabla_{b_2} = \sum \delta_z,
$$
$$
\delta_h = \delta_z \, W_2^\top \odot \mathbb{1}[X W_1 + b_1 > 0], \quad \nabla_{W_1} = X^\top \delta_h, \quad \nabla_{b_1} = \mathbf{1}^\top \delta_h.
$$

In [ ]:
def sigmoid(z):
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

def bce_loss(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

def mlp_init(D, H, seed):
    rng = np.random.default_rng(seed)
    W1 = rng.standard_normal((D, H)) * np.sqrt(2.0 / D)
    b1 = np.zeros(H)
    W2 = rng.standard_normal(H) * np.sqrt(2.0 / H)
    b2 = 0.0
    return W1, b1, W2, b2

def mlp_forward(X, W1, b1, W2, b2):
    pre  = X @ W1 + b1
    h    = np.maximum(pre, 0)
    z    = h @ W2 + b2
    return pre, h, z

def mlp_grads(X, y, W1, b1, W2, b2):
    N = X.shape[0]
    pre, h, z = mlp_forward(X, W1, b1, W2, b2)
    p = sigmoid(z)
    dz = (p - y) / N
    gW2 = h.T @ dz
    gb2 = dz.sum()
    dh = np.outer(dz, W2)
    dh[pre <= 0] = 0
    gW1 = X.T @ dh
    gb1 = dh.sum(axis=0)
    return gW1, gb1, gW2, gb2, p

def mlp_predict(X, W1, b1, W2, b2):
    _, _, z = mlp_forward(X, W1, b1, W2, b2)
    return sigmoid(z)

### Optimizers — one instance per parameter group

Reuse the same update rules from notebook 03. Each optimizer is instantiated once per tensor (W1, b1, W2, b2).

In [ ]:
class SGD:
    name = 'SGD'
    def __init__(self, shape):
        pass
    def step(self, g, lr):
        return -lr * g

class Momentum:
    name = 'Momentum'
    def __init__(self, shape, mu=0.9):
        self.mu = mu
        self.v = np.zeros(shape)
    def step(self, g, lr):
        self.v = self.mu * self.v + g
        return -lr * self.v

class NAG:
    name = 'NAG'
    def __init__(self, shape, mu=0.9):
        self.mu = mu
        self.v = np.zeros(shape)
    def step(self, g, lr):
        v_new = self.mu * self.v - lr * g
        delta = self.mu * v_new - lr * g
        self.v = v_new
        return delta

class AdaGrad:
    name = 'AdaGrad'
    def __init__(self, shape, eps=1e-8):
        self.eps = eps
        self.G = np.zeros(shape)
    def step(self, g, lr):
        self.G = self.G + g * g
        return -lr * g / (np.sqrt(self.G) + self.eps)

class RMSprop:
    name = 'RMSprop'
    def __init__(self, shape, beta=0.9, eps=1e-8):
        self.beta = beta; self.eps = eps
        self.E = np.zeros(shape)
    def step(self, g, lr):
        self.E = self.beta * self.E + (1 - self.beta) * g * g
        return -lr * g / (np.sqrt(self.E) + self.eps)

class Adam:
    name = 'Adam'
    def __init__(self, shape, b1=0.9, b2=0.999, eps=1e-8):
        self.b1, self.b2, self.eps = b1, b2, eps
        self.m = np.zeros(shape); self.v = np.zeros(shape); self.t = 0
    def step(self, g, lr):
        self.t += 1
        self.m = self.b1 * self.m + (1 - self.b1) * g
        self.v = self.b2 * self.v + (1 - self.b2) * g * g
        mh = self.m / (1 - self.b1 ** self.t)
        vh = self.v / (1 - self.b2 ** self.t)
        return -lr * mh / (np.sqrt(vh) + self.eps)

class AdamW(Adam):
    name = 'AdamW'
    def __init__(self, shape, wd=1e-4, **kw):
        super().__init__(shape, **kw)
        self.wd = wd

OPTIMIZERS = [SGD, Momentum, NAG, AdaGrad, RMSprop, Adam, AdamW]

### Training loop

In [ ]:
def train_mlp(opt_cls, lr, X, y, X_val, y_val,
              hidden=64, epochs=15, batch_size=2048, seed=SEED, wd=1e-4):
    D = X.shape[1]
    W1, b1, W2, b2 = mlp_init(D, hidden, seed=seed)

    def make(shape):
        return opt_cls(shape, wd=wd) if opt_cls is AdamW else opt_cls(shape)
    opt_W1, opt_b1 = make(W1.shape), make(b1.shape)
    opt_W2, opt_b2 = make(W2.shape), make(())

    rng = np.random.default_rng(seed + 1)
    N = X.shape[0]
    n_batches = int(np.ceil(N / batch_size))
    hist = {'epoch': [], 'train_loss': [], 'val_auc': []}

    for ep in range(epochs):
        idx = rng.permutation(N)
        ep_loss = 0.0
        for bi in range(n_batches):
            batch = idx[bi*batch_size:(bi+1)*batch_size]
            Xb, yb = X[batch], y[batch]
            gW1, gb1, gW2, gb2, p = mlp_grads(Xb, yb, W1, b1, W2, b2)
            dW1 = opt_W1.step(gW1, lr)
            db1 = opt_b1.step(gb1, lr)
            dW2 = opt_W2.step(gW2, lr)
            db2_arr = opt_b2.step(np.array(gb2), lr)
            if opt_cls is AdamW:
                dW1 = dW1 - lr * opt_W1.wd * W1
                dW2 = dW2 - lr * opt_W2.wd * W2
            W1 += dW1; b1 += db1; W2 += dW2; b2 += float(db2_arr)
            ep_loss += bce_loss(yb, p) * Xb.shape[0]
        ep_loss /= N
        val_auc = roc_auc_score(y_val, mlp_predict(X_val, W1, b1, W2, b2))
        hist['epoch'].append(ep + 1)
        hist['train_loss'].append(ep_loss)
        hist['val_auc'].append(val_auc)
    return (W1, b1, W2, b2), hist

### Smoke test — Adam, 5 epochs

In [ ]:
t0 = time.time()
_, smoke = train_mlp(Adam, 0.005, X_tr, y_tr, X_val, y_val, hidden=64, epochs=5)
print(f'smoke: {time.time()-t0:.1f}s   ',
      'train_loss:', [f'{x:.4f}' for x in smoke['train_loss']],
      '  val_auc:', [f'{x:.4f}' for x in smoke['val_auc']])

### Learning-rate sweep across all 7 optimizers

In [ ]:
LR_GRID = {
    'SGD'      : [0.01, 0.05, 0.1, 0.5, 1.0],
    'Momentum' : [0.001, 0.005, 0.01, 0.05, 0.1],
    'NAG'      : [0.001, 0.005, 0.01, 0.05, 0.1],
    'AdaGrad'  : [0.01, 0.05, 0.1, 0.5, 1.0],
    'RMSprop'  : [0.0001, 0.0005, 0.001, 0.005, 0.01],
    'Adam'     : [0.0005, 0.001, 0.005, 0.01, 0.05],
    'AdamW'    : [0.0005, 0.001, 0.005, 0.01, 0.05],
}
EPOCHS  = 15
BATCH   = 2048
HIDDEN  = 64

sweep_runs = []
for opt_cls in OPTIMIZERS:
    print(f'\n=== {opt_cls.name} ===')
    for lr in LR_GRID[opt_cls.name]:
        t0 = time.time()
        try:
            _, hist = train_mlp(opt_cls, lr, X_tr, y_tr, X_val, y_val,
                                 hidden=HIDDEN, epochs=EPOCHS, batch_size=BATCH)
            elapsed = time.time() - t0
            print(f'  lr={lr:<8g} final_auc={hist["val_auc"][-1]:.5f}  '
                  f'best_auc={max(hist["val_auc"]):.5f}  t={elapsed:.1f}s')
            sweep_runs.append({'optimizer': opt_cls.name, 'lr': lr,
                                'final_auc': hist['val_auc'][-1],
                                'best_auc': max(hist['val_auc']),
                                'time_s': elapsed, 'hist': hist})
        except (FloatingPointError, ValueError) as e:
            print(f'  lr={lr:<8g} DIVERGED ({type(e).__name__})')
            sweep_runs.append({'optimizer': opt_cls.name, 'lr': lr,
                                'final_auc': float('nan'),
                                'best_auc': float('nan'),
                                'time_s': float('nan'), 'hist': None})

In [ ]:
sweep_df = pd.DataFrame([{k:v for k,v in r.items() if k!='hist'} for r in sweep_runs])
print(sweep_df.pivot_table(index='optimizer', columns='lr', values='best_auc'))

best = (sweep_df.dropna(subset=['best_auc'])
        .sort_values('best_auc', ascending=False)
        .drop_duplicates('optimizer', keep='first'))
print('\n--- best lr per optimizer ---')
print(best[['optimizer','lr','best_auc','final_auc','time_s']].to_string(index=False))

### Convergence plot (best lr each)

In [ ]:
best_runs = {row.optimizer: [r for r in sweep_runs
                              if r['optimizer']==row.optimizer and r['lr']==row.lr][0]
             for row in best.itertuples()}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = plt.cm.tab10(np.linspace(0, 1, len(best_runs)))
for (name, run), c in zip(best_runs.items(), colors):
    h = run['hist']
    axes[0].plot(h['epoch'], h['train_loss'], label=f'{name}  lr={run["lr"]:g}', color=c)
    axes[1].plot(h['epoch'], h['val_auc'],    label=f'{name}  lr={run["lr"]:g}', color=c)
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('train BCE')
axes[0].set_title('Shallow NN — training loss')
axes[0].set_yscale('log'); axes[0].grid(alpha=0.3); axes[0].legend(fontsize=8)
axes[1].axhline(LBFGS_AUC, ls='--', color='black', alpha=0.6,
                label=f'L-BFGS linear ref = {LBFGS_AUC:.4f}')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('val AUC')
axes[1].set_title('Shallow NN — validation AUC')
axes[1].grid(alpha=0.3); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(SPLITS_DIR, 'fig_mlp_optimizer_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

### Save MLP results

In [ ]:
summary_rows = []
thr = LBFGS_AUC - 0.001
for name, run in best_runs.items():
    h = run['hist']
    hit = [e for e, a in zip(h['epoch'], h['val_auc']) if a >= thr]
    summary_rows.append({
        'optimizer': name,
        'best_lr'  : run['lr'],
        'final_train_loss': h['train_loss'][-1],
        'final_val_auc'   : h['val_auc'][-1],
        'best_val_auc'    : run['best_auc'],
        'epochs_to_linear_baseline': hit[0] if hit else None,
        'time_s'          : run['time_s'],
    })
summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))

out = {
    'config': {'epochs': EPOCHS, 'batch_size': BATCH, 'hidden': HIDDEN,
                'seed': SEED, 'lr_grid': LR_GRID, 'lbfgs_ref_auc': LBFGS_AUC},
    'sweep' : [{k:v for k,v in r.items() if k!='hist'} for r in sweep_runs],
    'best'  : summary.to_dict(orient='records'),
    'curves': {n: {'epoch': r['hist']['epoch'],
                    'train_loss': r['hist']['train_loss'],
                    'val_auc': r['hist']['val_auc'],
                    'best_lr': r['lr']}
                for n, r in best_runs.items()},
}
with open(os.path.join(SPLITS_DIR, 'results_mlp_optimizers.json'), 'w') as fh:
    json.dump(out, fh, indent=2, default=float)
summary.to_csv(os.path.join(SPLITS_DIR, 'results_mlp_optimizers_summary.csv'), index=False)
print('\nsaved results_mlp_optimizers.json and summary.csv')